In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# Download and extract datasets (same as before)
dataset_url = 'http://www.cs.toronto.edu/~vendrov/order/coco.zip'
dataset_path = 'coco.zip'
data_dir = 'coco_data'

if not os.path.exists(data_dir):
    print("Downloading COCO dataset...")
    urllib.request.urlretrieve(dataset_url, dataset_path)
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
print("Dataset ready.")

glove_url = 'http://nlp.stanford.edu/data/glove.6B.zip'
glove_zip = 'glove.6B.zip'
glove_dir = 'glove_data'

if not os.path.exists(glove_dir):
    print("Downloading GloVe embeddings...")
    urllib.request.urlretrieve(glove_url, glove_zip)
    with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
        zip_ref.extractall(glove_dir)
print("GloVe embeddings ready.")

Dataset ready.
GloVe embeddings ready.


In [ ]:
def load_subset(data_dir, subset=5000):
    """Load subset of data"""
    def read_txt(path):
        with open(path, 'r') as f:
            return [line.strip() for line in f.readlines()[:subset]]

    caps_dir = os.path.join(data_dir, 'data', 'coco')
    train_caps = read_txt(os.path.join(caps_dir, 'train.txt'))
    val_caps = read_txt(os.path.join(caps_dir, 'val.txt'))
    test_caps = read_txt(os.path.join(caps_dir, 'test.txt'))

    train_feats = np.load(os.path.join(caps_dir, 'images', '10crop', 'train.npy'))[:subset]
    val_feats = np.load(os.path.join(caps_dir, 'images', '10crop', 'val.npy'))[:subset]
    test_feats = np.load(os.path.join(caps_dir, 'images', '10crop', 'test.npy'))[:subset]

    return (train_caps, train_feats), (val_caps, val_feats), (test_caps, test_feats)

# Load data
(train_caps, train_feats), (val_caps, val_feats), (test_caps, test_feats) = load_subset(data_dir)
print(f"Loaded {len(train_caps)} training samples")

Loaded 5000 training samples


In [ ]:
def build_vocab(captions, min_freq=2):
    """Build vocabulary from captions"""
    tokens = [word for sent in captions for word in sent.split()]
    counter = Counter(tokens)
    vocab = {'<pad>': 0, '<unk>': 1}
    idx = 2
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

vocab = build_vocab(train_caps)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Max vocab index: {max(vocab.values())}")

Vocab size: 1859
Max vocab index: 1858


In [ ]:
def captions_to_indices(captions, vocab, max_len=15):
    """Convert captions to token indices"""
    data = []
    for sent in captions:
        idxs = [vocab.get(w, vocab['<unk>']) for w in sent.split()[:max_len]]
        idxs += [vocab['<pad>']] * (max_len - len(idxs))
        data.append(idxs)
    return torch.LongTensor(data)

# Prepare data tensors
train_seqs = captions_to_indices(train_caps, vocab).to(device)
val_seqs = captions_to_indices(val_caps, vocab).to(device)
test_seqs = captions_to_indices(test_caps, vocab).to(device)
train_feats = torch.FloatTensor(train_feats).to(device)
val_feats = torch.FloatTensor(val_feats).to(device)
test_feats = torch.FloatTensor(test_feats).to(device)

In [ ]:
def load_glove_matrix(glove_path, vocab, embed_dim=300):
    """Load GloVe embeddings for vocabulary"""
    max_vocab_idx = max(vocab.values()) if vocab else 0
    embeddings = np.random.randn(max_vocab_idx + 1, embed_dim) * 0.01  # Small random init
    embeddings[0] = 0  # Padding token

    found = 0
    with open(glove_path, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            if word in vocab:
                vec = np.array(list(map(float, parts[1:])))
                embeddings[vocab[word]] = vec
                found += 1
    print(f"Found GloVe embeddings for {found}/{len(vocab)} words")
    return embeddings

glove_path = 'glove_data/glove.6B.300d.txt'
glove_matrix = load_glove_matrix(glove_path, vocab)

Found GloVe embeddings for 1853/1859 words


In [ ]:
class JointEmbeddingModel(nn.Module):
    """Base class for joint embedding models"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings=None):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Word embeddings
        # Initialize embedding layer with vocab_size
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if glove_embeddings is not None:
            # If GloVe embeddings are provided, load them into the embedding layer
            # Ensure glove_embeddings is a torch tensor and on the correct device
            glove_tensor = torch.FloatTensor(glove_embeddings).to(self.embedding.weight.device)
            # Check if the size matches, if not, handle potential mismatch
            if self.embedding.weight.size() == glove_tensor.size():
                 self.embedding.weight = nn.Parameter(glove_tensor)
            else:
                 print("Warning: GloVe embedding size does not match initialized embedding layer size.")
                 print(f"Initialized size: {self.embedding.weight.size()}, GloVe size: {glove_tensor.size()}")
                 # Handle mismatch - e.g., copy matching embeddings and keep random init for others
                 min_rows = min(self.embedding.weight.size(0), glove_tensor.size(0))
                 self.embedding.weight.data[:min_rows, :] = glove_tensor[:min_rows, :]


        # Image projection layer (W_i in the paper)
        self.img_proj = nn.Linear(img_feat_dim, hidden_dim)

    def forward_caption(self, captions):
        """Encode captions to embedding space"""
        raise NotImplementedError

    def forward_image(self, images):
        """Project images to embedding space"""
        return self.img_proj(images)

    def forward(self, captions, images):
        """Forward pass for both modalities"""
        capt_emb = self.forward_caption(captions)
        img_emb = self.forward_image(images)
        return capt_emb, img_emb

In [ ]:
class GRUJointEmbedding(JointEmbeddingModel):
    """GRU-based joint embedding model"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings=None):
        super().__init__(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward_caption(self, captions):
        emb = self.embedding(captions)
        _, h = self.gru(emb)
        return h.squeeze(0)


class LSTMJointEmbedding(JointEmbeddingModel):
    """LSTM-based joint embedding model"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings=None):
        super().__init__(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward_caption(self, captions):
        emb = self.embedding(captions)
        _, (h, _) = self.lstm(emb)
        return h.squeeze(0)


class BiGRUJointEmbedding(JointEmbeddingModel):
    """Bidirectional GRU-based joint embedding model"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings=None):
        super().__init__(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_embeddings)
        self.bigru = nn.GRU(embed_dim, hidden_dim // 2, batch_first=True, bidirectional=True)

    def forward_caption(self, captions):
        emb = self.embedding(captions)
        _, h = self.bigru(emb)
        # Concatenate forward and backward hidden states
        h = torch.cat([h[0], h[1]], dim=1)
        return h

In [ ]:
def pairwise_ranking_loss(capt_embs, img_embs, margin=0.2):
    """Pairwise ranking loss with normalization"""
    # Normalize embeddings for cosine similarity
    capt_embs = nn.functional.normalize(capt_embs, p=2, dim=1)
    img_embs = nn.functional.normalize(img_embs, p=2, dim=1)

    # Compute similarity matrix
    scores = torch.matmul(capt_embs, img_embs.t())

    # Get diagonal (positive pairs)
    diagonal = scores.diag().view(capt_embs.size(0), 1)

    # Caption-to-image loss
    cost_s = (margin + scores - diagonal).clamp(min=0)

    # Image-to-caption loss
    cost_im = (margin + scores - diagonal.t()).clamp(min=0)

    # Mask out diagonal
    mask = torch.eye(scores.size(0), device=device).bool()
    cost_s = cost_s.masked_fill(mask, 0)
    cost_im = cost_im.masked_fill(mask, 0)

    # Average instead of sum to normalize by batch size
    return (cost_s.mean() + cost_im.mean())


def train_model(model, captions, images, val_captions=None, val_images=None,
                epochs=30, batch_size=128, lr=0.001, patience=5):
    """Train model with early stopping"""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(captions, images)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch_c, batch_i in dataloader:
            optimizer.zero_grad()
            capt_emb, img_emb = model(batch_c, batch_i)
            loss = pairwise_ranking_loss(capt_emb, img_emb)
            loss.backward()

            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)

        # Validation
        if val_captions is not None and val_images is not None:
            model.eval()
            with torch.no_grad():
                val_dataset = TensorDataset(val_captions, val_images)
                val_loader = DataLoader(val_dataset, batch_size=batch_size)
                val_loss = 0
                for batch_c, batch_i in val_loader:
                    capt_emb, img_emb = model(batch_c, batch_i)
                    val_loss += pairwise_ranking_loss(capt_emb, img_emb).item()
                val_loss /= len(val_loader)

            print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {avg_loss:.4f}, Val Loss: {val_loss:.4f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break
        else:
            print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")


# ==================== IMPROVED EVALUATION ====================

def evaluate_model(model, captions, images, captions_per_image=5, batch_size=64):
    """Evaluate model using Recall@K metrics"""
    model.eval()

    capt_embs_list = []
    img_embs_list = []

    dataset = TensorDataset(captions, images)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch_c, batch_i in dataloader:
            capt_emb, img_emb = model(batch_c, batch_i)
            capt_embs_list.append(capt_emb)
            # Only keep one embedding per image (every 5th)
            img_embs_list.append(img_emb[::captions_per_image])

    capt_embs = torch.cat(capt_embs_list, dim=0)
    img_embs = torch.cat(img_embs_list, dim=0)

    # Normalize for cosine similarity
    capt_embs = nn.functional.normalize(capt_embs, p=2, dim=1)
    img_embs = nn.functional.normalize(img_embs, p=2, dim=1)

    # Compute similarity matrix
    sim_matrix = torch.matmul(capt_embs, img_embs.t())

    # Calculate metrics
    ranks = []
    num_captions = sim_matrix.shape[0]

    for i in range(num_captions):
        sorted_indices = torch.argsort(sim_matrix[i], descending=True)
        true_img_index = i // captions_per_image
        rank = (sorted_indices == true_img_index).nonzero(as_tuple=False).item()
        ranks.append(rank)

    ranks = torch.tensor(ranks, dtype=torch.float32)
    recall_1 = (ranks < 1).float().mean().item() * 100
    recall_5 = (ranks < 5).float().mean().item() * 100
    recall_10 = (ranks < 10).float().mean().item() * 100
    mean_rank = ranks.mean().item() + 1
    median_rank = ranks.median().item() + 1

    return recall_1, recall_5, recall_10, mean_rank, median_rank

In [ ]:
# Hyperparameters (optimized based on paper)
embed_dim = 300
hidden_dim = 1024
img_feat_dim = 4096
epochs = 30
batch_size = 128
lr = 0.001

print("\n" + "="*60)
print("TRAINING MODELS")
print("="*60)

# Add this print statement to check the maximum index
print(f"Max index in train_seqs: {torch.max(train_seqs)}")
print(f"Vocab size: {vocab_size}")


# 1. GRU without GloVe
print("\n1. Training GRU (from scratch):")
gru_model = GRUJointEmbedding(vocab_size, embed_dim, hidden_dim, img_feat_dim)
train_model(gru_model, train_seqs, train_feats, val_seqs, val_feats,
            epochs=epochs, batch_size=batch_size, lr=lr)

# 2. GRU with GloVe
print("\n2. Training GRU (with GloVe):")
gru_glove_model = GRUJointEmbedding(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_matrix)
train_model(gru_glove_model, train_seqs, train_feats, val_seqs, val_feats,
            epochs=epochs, batch_size=batch_size, lr=lr)

# 3. LSTM with GloVe
print("\n3. Training LSTM (with GloVe):")
lstm_glove_model = LSTMJointEmbedding(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_matrix)
train_model(lstm_glove_model, train_seqs, train_feats, val_seqs, val_feats,
            epochs=epochs, batch_size=batch_size, lr=lr)

# 4. Bidirectional GRU with GloVe
print("\n4. Training Bidirectional GRU (with GloVe):")
bigru_glove_model = BiGRUJointEmbedding(vocab_size, embed_dim, hidden_dim, img_feat_dim, glove_matrix)
train_model(bigru_glove_model, train_seqs, train_feats, val_seqs, val_feats,
            epochs=epochs, batch_size=batch_size, lr=lr)

print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

models = [
    ("GRU (from scratch)", gru_model),
    ("GRU + GloVe", gru_glove_model),
    ("LSTM + GloVe", lstm_glove_model),
    ("BiGRU + GloVe", bigru_glove_model)
]

print("\n{:<25} {:>8} {:>8} {:>9} {:>10} {:>12}".format(
    "Model", "R@1", "R@5", "R@10", "Mean Rank", "Median Rank"))
print("-" * 80)

for name, model in models:
    r1, r5, r10, mean_r, median_r = evaluate_model(model, test_seqs, test_feats)
    print("{:<25} {:>7.1f}% {:>7.1f}% {:>8.1f}% {:>10.1f} {:>12.1f}".format(
        name, r1, r5, r10, mean_r, median_r))

print("\n" + "="*60)
print("Target Results from Paper:")
print("GRU (Vendrov et al.): R@1=36.3%, R@10=85.8%, Mean Rank=7.6")
print("GRU + GloVe: R@1=37.0%, R@10=86.8%, Mean Rank=7.3")
print("LSTM + GloVe: R@1=35.4%, R@10=86.2%, Mean Rank=7.5")
print("="*60)


TRAINING MODELS
Max index in train_seqs: 1858
Vocab size: 1859

1. Training GRU (from scratch):
Epoch 1/30 - Train Loss: 0.3853, Val Loss: 0.3950
Epoch 2/30 - Train Loss: 0.3066, Val Loss: 0.4172
Epoch 3/30 - Train Loss: 0.2181, Val Loss: 0.4266
Epoch 4/30 - Train Loss: 0.1374, Val Loss: 0.4320
Epoch 5/30 - Train Loss: 0.0757, Val Loss: 0.4283
Epoch 6/30 - Train Loss: 0.0362, Val Loss: 0.4233
Early stopping at epoch 6

2. Training GRU (with GloVe):
Epoch 1/30 - Train Loss: 0.3769, Val Loss: 0.3931
Epoch 2/30 - Train Loss: 0.3014, Val Loss: 0.4044
Epoch 3/30 - Train Loss: 0.2343, Val Loss: 0.4194
Epoch 4/30 - Train Loss: 0.1699, Val Loss: 0.4271
Epoch 5/30 - Train Loss: 0.1111, Val Loss: 0.4355
Epoch 6/30 - Train Loss: 0.0712, Val Loss: 0.4322
Early stopping at epoch 6

3. Training LSTM (with GloVe):
Epoch 1/30 - Train Loss: 0.3822, Val Loss: 0.3949
Epoch 2/30 - Train Loss: 0.3184, Val Loss: 0.4078
Epoch 3/30 - Train Loss: 0.2656, Val Loss: 0.4050
Epoch 4/30 - Train Loss: 0.2172, Val L